# Background

- **Author**: 詹明翰
- **Created At**: 2026-10-16
- **Research Motivation and Context (why are we interested in the findings?)：**
  描述114年臺北市死傷交通事故的時間、地區、照明與道路環境，先交叉檢視照明設備狀態與日夜，
  再比較夜間不同照明組的傷亡情形，為後續關聯研究提供基礎。
  輸入為 `data/processed/taipei_2025_descriptive.csv`；[資料定義](#data-description)與完整摘要均在本檔。
- **Main Findings and Takeaways：**
  全年22,762筆；日出日落夜間6,795筆，其中照明未開啟或故障363筆、無照明97筆，合計460筆。
  目標事故占完整樣本2.02%，占照明已知夜間事故6.77%。夜間三組樣本大小及環境構成不同，
  描述上的死亡比例差異不能視為照明的因果效果。
- **Future Direction：**
  核對官方事故識別碼與速限0的意義，補充重傷及交通暴露資料，再評估控制道路與天候後的關聯。

本文以原始第一當事人紀錄暫作每列一事故，使用全年樣本，依各項分析需要處理缺值。
每個比例均列出分母，死亡事故統一定義為「24小時內死亡人數≥1」。

## 閱讀目錄

| 章節 | 內容 |
|---|---|
| [1. 資料與研究定義](#data-description) | 來源、觀察單位、欄位、日出日落與固定夜間定義、資料預覽 |
| [2. 樣本與變數分布](#variable-summaries) | 傷亡及速限數值摘要；月份、行政區、照明、天候、道路型態與兩種夜間指標的完整分布 |
| [3. 照明設備狀態與日夜交叉分析](#lighting-night) | 3×2件數、六格占比、目標事故比例與分母 |
| [4. 夜間不同照明條件比較](#night-comparison) | 傷亡、速限、天候、道路型態，以及主要兩組的平均差 |
| [5. 夜間定義敏感度](#night-sensitivity) | 日出日落與固定18:49–05:02兩種樣本的比較 |
| [6. 研究重點與簡報引用](#presentation-brief) | 重點數字、單位與分母，及結果解讀 |
| [7. 研究限制與後續工作](#limitations) | 缺值、定義、樣本量與因果解讀限制 |
| [8. 資料驗證與輸出檔案](#data-checks) | 可執行檢查、執行狀態與CSV索引 |

製作簡報時可依此順序閱讀，也可先看第6節；引用數字時請保留樣本、分母與單位。

<a id="data-description"></a>

## 1. 資料與研究定義

### 1.1 Data Description

- **Background**：臺北市114年（2025年）死傷交通事故明細，代碼依 [事故對照表.pdf](../data/raw/事故對照表.pdf)。
- **Source data**：[原始CSV](../data/raw/114年-臺北市死傷交通事故明細.csv)，原始47欄完整保留。
- **Unit of observation**：第一當事人紀錄。全部 `當事人序號=1`，完整列與「年月日時分＋肇事地點＋座標」候選鍵無重複。
  缺少官方事故識別碼，因此暫以一列一事故，並非官方確認的事故唯一鍵。
- **Sample period**：2025-01-01至2025-12-31，涵蓋12個月與12個行政區，共22,762筆。
- **Data path / Role**：`data/processed/taipei_2025_descriptive.csv`；analysis。
- **Date first created**：2026-10-16（衍生檔建立日期）。
- **Processing code**：[prepare_descriptive_statistics.ipynb](../data/code/prepare_descriptive_statistics.ipynb)。
- **Schema**：58欄，即47個原始欄位與11個衍生欄位；列數不增減。
- **SHA-256**：`09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60`。
- **Known issues**：保留原始缺值；速限0意義未確認；沒有可用的重傷分類；缺座標者無法計算主要夜間指標。

### 1.2 Definition for each variable

原始欄位的完整代碼見PDF；本次使用的原始變數包括事故年月日時分（民國年與當地時間）、區序與肇事地點、
死亡人數（24小時內）、2–30日死亡人數、受傷人數、當事人序號、光線、天候、道路型態、速限（km/h）與經緯度。
數字代碼依類別意義摘要，不直接計算代碼平均。

| 衍生欄位 | 定義 |
|---|---|
| `record_id` | 原始列順序1–22,762，供追溯，非官方事故編號 |
| `crash_datetime` | 民國年加1911，組合日期時間；Asia/Taipei當地時間 |
| `district` | 移除區序開頭數字所得行政區 |
| `lighting_label`、`weather_label`、`road_form_label` | PDF中文標籤；缺值保留 |
| `fatal24` | 24小時內死亡人數≥1為1，已知為0則為0，原始缺值保留 |
| `sunrise_local`、`sunset_local` | Astral 3.2以日期、座標與海拔0公尺計算，含UTC+8時區的ISO時間 |
| `is_night` | 早於當日日出或晚於當日日落為1，其餘0；缺座標為缺值 |
| `night_fixed` | 18:49（含）至翌日05:02（不含）為1，其餘0；0不代表白天 |

<a id="night-definition"></a>

### 1.3 夜間判定的計算來源與定義

**資料來源與方法。** 本研究沒有將中央氣象署公布的日出日落表匯入資料。
日出與日落時間由 Python 的 **Astral 3.2** 計算：輸入每筆事故的西元日期、
`座標-Y`（緯度）與 `座標-X`（經度），海拔統一設為 0 公尺，使用臺灣時間 UTC+8
（2025 年的 Asia/Taipei）。套件依天文模型推算太陽位置及日出日落，並非現場亮度量測。

**主要指標 `is_night`。** 令事故的當地時間為 t、該日該位置的日出與日落為 R、S：
當 **t < R 或 t > S** 時設為 1，其餘為 0；恰好等於日出或日落歸為 0。
若經度或緯度缺失，日出日落及主要夜間指標都保留缺值，不用市中心座標代替。
本資料有 2 筆這類紀錄。`光線` 是照明設備狀態，不能直接取代這個日夜判斷。

**固定指標 `night_fixed`。** 採「全年最晚日落之後、
全年最早日出之前」作為保守的固定時段。先對 **2025 年完整 365 日**，
在資料全部有效座標的經緯度外接矩形四角計算日出日落，涵蓋本資料中的位置差異；
經度範圍為 121.4611803–121.6520178、緯度為 24.9659305–25.1832948。
這是本資料的空間範圍，並非臺北市官方行政邊界。

四角全年計算的最早日出為 **2025-06-09 05:02:50.800739**（東北角），
最晚日落為 **2025-07-02 18:48:12.258035**（西北角）。配合事故時間只有分鐘精度，
將最早日出向下取整為 05:02、最晚日落向上取整為 18:49。
故 `night_fixed = 1` 的時段為 **18:49（含）至翌日 05:02（不含）**，
即一天中的分鐘數 m ≥ 1129 或 m < 302，其餘設為 0。
另外逐筆驗證：所有有有效座標且被固定時段納入的事故，必須同時符合其當日 `is_night = 1`。

這個固定指標是較窄的夜間子樣本；值為 0 只表示不在固定時段內，**不代表白天**。
固定時段依時間即可套用；但缺座標者無法逐筆驗證實際位置，所以「皆為夜間」的檢查
限定於有有效座標的紀錄，不能延伸到其他年度、其他地區或未知位置。

**為何不只採夏至？** 夏至白晝最長，並不保證當天是全年日出最早或日落最晚。
此點參考 [臺北市立天文科學教育館的 2025 夏至說明](https://tam.gov.taipei/News_Content.aspx?n=B64052C7930D4913&s=C881A8681CC67490&sms=2CF1F5E2E0B96411)。
該網頁用於確認天文概念；本資料各筆時間與固定界線仍由 Astral 計算，不冒稱為官方公布值。

**解讀限制。** 日落後與日出前可能仍有暮光或曙光；天候、山勢、建築遮蔽與實際照明
也會影響亮度。因此本定義表示模型下的日出日落夜間，不能稱為「完全黑暗」。

<a id="data-sample"></a>

### 1.4 讀取資料與樣本預覽

以下設定可攜式專案路徑，核對固定SHA-256、列數、欄數及必要欄位，再檢視前十筆。
前十筆用於確認內容，並非隨機抽樣，不能代表全年分布。

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root():
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / 'AGENTS.md').is_file() and (p / 'templates').is_dir():
            return p
    raise FileNotFoundError('請從含 AGENTS.md 與 templates/ 的專案內開啟 Notebook。')

PROJECT_ROOT = find_project_root()

def project_file(relative, area, must_exist=True):
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('請使用精確的專案相對路徑。')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data' / area).resolve()):
        raise ValueError('資料路徑超出預期目錄。')
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def verify(path, expected):
    actual = sha256_file(path)
    if actual != expected:
        raise ValueError(f'檔案版本改變，請核對後再執行：{path.name}')
    return actual

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [2]:
INPUT_RELATIVE_PATH = 'data/processed/taipei_2025_descriptive.csv'
EXPECTED_SHA256 = '09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60'
data_path = project_file(INPUT_RELATIVE_PATH, 'processed')
verify(data_path, EXPECTED_SHA256)
df = pd.read_csv(data_path, encoding='utf-8-sig')
assert df.shape == (22762, 58)
assert df['record_id'].is_unique
required = ['crash_datetime','district','lighting_label','weather_label','road_form_label',
            '死亡人數','2-30日死亡人數','受傷人數','速限-速度限制','fatal24','is_night','night_fixed']
assert set(required).issubset(df.columns)
dt = pd.to_datetime(df['crash_datetime'])

In [3]:
print({'rows':len(df),'columns':len(df.columns),'sha256':sha256_file(data_path)})

{'rows': 22762, 'columns': 58, 'sha256': '09c8bae171a8307414aa787ce36f9d5d1f2f1c455dd619e250ac3fae41f15a60'}


In [4]:
display(df[required].head(10))

,crash_datetime,district,lighting_label,weather_label,road_form_label,死亡人數,2-30日死亡人數,受傷人數,速限-速度限制,fatal24,is_night,night_fixed
0,2025-01-01 00:18:00,中山區,有照明且開啟,晴,直路,0.0000,0.0000,1.0000,40.0000,0.0000,1.0000,1
1,2025-01-01 00:31:00,文山區,有照明且開啟,陰,直路,0.0000,0.0000,3.0000,50.0000,0.0000,1.0000,1
2,2025-01-01 00:35:00,中山區,有照明且開啟,陰,多岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
3,2025-01-01 00:41:00,中山區,有照明且開啟,陰,直路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
4,2025-01-01 00:57:00,南港區,有照明且開啟,陰,三岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
5,2025-01-01 01:34:00,信義區,有照明且開啟,陰,三岔路,0.0000,0.0000,3.0000,30.0000,0.0000,1.0000,1
6,2025-01-01 02:15:00,信義區,有照明且開啟,陰,直路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,1
7,2025-01-01 06:26:00,文山區,無照明,晴,坡路,0.0000,0.0000,1.0000,30.0000,0.0000,1.0000,0
8,2025-01-01 06:29:00,文山區,有照明未開啟或故障,晴,三岔路,0.0000,0.0000,1.0000,50.0000,0.0000,1.0000,0
9,2025-01-01 07:11:00,松山區,有照明且開啟,晴,直路,0.0000,0.0000,2.0000,30.0000,0.0000,0.0000,0


<a id="variable-summaries"></a>

## 2. 樣本與變數分布

### 2.1 傷亡與速限的數值摘要

平均、標準差與分位數均以各欄非缺值紀錄計算，缺值N另列，不以0取代。
死亡人數的平均是每件平均死亡人數，`fatal24`的平均則是死亡事故比例；兩者意義不同。
受傷人數是每件事故的受傷總人數，不是第一當事人的傷勢。速限是道路限制，不是事故當下車速。
中位數與第25／75百分位描述分布集中範圍，最大值則反映少數較大的觀測。

In [5]:
def show(title, frame):
    display(Markdown('### ' + title))
    display(frame)

numeric = ['死亡人數','2-30日死亡人數','受傷人數','速限-速度限制','fatal24']
stats = df[numeric].describe(percentiles=[.25,.5,.75]).T
stats = stats.rename(columns={'count':'有效 N','mean':'平均','std':'標準差','min':'最小',
                              '25%':'第25百分位','50%':'中位數','75%':'第75百分位','max':'最大'})
stats['缺值 N'] = df[numeric].isna().sum()
stats.index = ['24小時內死亡人數','2–30日死亡人數','受傷人數','速限（km/h；保留0）','24小時內死亡事故指標（0/1）']
show(f'傷亡與速限數值摘要（全樣本 N={len(df):,}）', stats)

### 傷亡與速限數值摘要（全樣本 N=22,762）

,有效 N,平均,標準差,最小,第25百分位,中位數,第75百分位,最大,缺值 N
24小時內死亡人數,"22,755.0000",0.0025,0.0513,0.0000,0.0000,0.0000,0.0000,2.0000,7
2–30日死亡人數,"22,752.0000",0.0001,0.0115,0.0000,0.0000,0.0000,0.0000,1.0000,10
受傷人數,"22,761.0000",1.3269,0.5970,0.0000,1.0000,1.0000,2.0000,21.0000,1
速限（km/h；保留0）,"22,758.0000",44.1460,10.0558,0.0000,40.0000,50.0000,50.0000,80.0000,4
24小時內死亡事故指標（0/1）,"22,755.0000",0.0025,0.0500,0.0000,0.0000,0.0000,0.0000,1.0000,7


受傷人數中位數為1、平均約1.327人，最大21人，表示少數事故涉及多名傷者。
24小時內死亡人數大多為0，須搭配死亡事故件數解讀；已知死亡58人來自57件事故。
速限中位數為50 km/h，原始0值保留，平均速限仍須配合分布與0值定義解讀。

下列七個類別變數各自分表，顯示全部已觀察到的類別。百分比分母為該變數全部非缺值紀錄，
缺值另外列出；有效百分比加總為100%（顯示值可能有四捨五入差異）。月份依時間排序，其餘依件數排序。

In [6]:
category_summaries = {}

def summarize_category(name, series, sort_by_category=False):
    valid_n = int(series.notna().sum())
    missing_n = int(series.isna().sum())
    counts = series.dropna().value_counts()
    if sort_by_category:
        counts = counts.sort_index()
    table = counts.rename_axis('類別').reset_index(name='筆數')
    table['有效樣本占比（%）'] = table['筆數'] / valid_n * 100 if valid_n else np.nan
    display(Markdown(f'**{name}**｜全樣本 N={len(series):,}；有效 N={valid_n:,}；缺值 N={missing_n:,}。'
                     f'百分比分母為 {valid_n:,} 筆有效紀錄。'))
    display(table)
    assert int(table['筆數'].sum()) == valid_n
    if valid_n:
        assert np.isclose(table['有效樣本占比（%）'].sum(), 100)
    category_summaries[name] = table
    return table

<a id="category-month"></a>

### 2.2 發生月份

觀察一年內事故紀錄如何分布。月份是週期性類別，這裡計算各月件數與比例，而不計算「平均月份」。月份天數及交通量不同，件數差異不能直接解讀為事故風險差異。

In [7]:
month_summary = summarize_category('發生月份', dt.dt.month, sort_by_category=True)

**發生月份**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,1,1896,8.3297
1,2,1641,7.2094
2,3,2020,8.8744
3,4,1855,8.1495
4,5,1867,8.2023
5,6,1773,7.7893
6,7,1851,8.1320
7,8,1949,8.5625
8,9,1929,8.4747
9,10,1951,8.5713


**結果解讀：** 資料涵蓋 1 至 12 月，沒有缺月。12 月件數最多（2,063 筆），2 月最少（1,641 筆）。這是月份分布，未依每月天數或交通量標準化。

<a id="category-district"></a>

### 2.3 行政區

行政區標籤由原始區序移除開頭數字而來。每列計入其記錄的行政區，百分比用來說明事故紀錄在臺北市各區的分布。

In [8]:
district_summary = summarize_category('行政區', df['district'])

**行政區**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,中山區,3015,13.2458
1,大安區,2495,10.9613
2,中正區,2174,9.5510
3,信義區,2097,9.2127
4,士林區,2074,9.1117
5,內湖區,2010,8.8305
6,萬華區,1684,7.3983
7,大同區,1639,7.2006
8,北投區,1592,6.9941
9,文山區,1504,6.6075


**結果解讀：** 12 個行政區都有紀錄，中山區最多（3,015 筆），南港區最少（1,019 筆）。區域面積、人口與交通量不同，不能只用此表判斷哪一區的道路較安全。

<a id="category-lighting"></a>

### 2.4 照明設備狀態

依 PDF 將光線代碼轉為設備狀態。代碼 6 同時包括「未開啟」與「故障」，原始資料無法再細分，因此這裡保持原類別。此表包含全天紀錄，不代表各組都是夜間事故。

In [9]:
lighting_summary = summarize_category('照明設備狀態', df['lighting_label'])

**照明設備狀態**｜全樣本 N=22,762；有效 N=22,750；缺值 N=12。百分比分母為 22,750 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,有照明未開啟或故障,9296,40.8615
1,有照明且開啟,9058,39.8154
2,無照明,4396,19.3231


**結果解讀：** 有效紀錄為 22,750 筆，另有 12 筆缺值。「有照明未開啟或故障」9,296 筆、「有照明且開啟」9,058 筆、「無照明」4,396 筆。夜間比較還須與夜間指標交叉篩選。 占全年22,762筆的比例依序為40.84%、39.79%、19.31%，另12筆缺值（0.05%）；下表百分比則以有效22,750筆為分母。

<a id="category-weather"></a>

### 2.5 天候

天候依 PDF 對照為晴、陰、雨等文字類別。各類別百分比描述事故發生時記錄的天候，沒有使用不同天候的行車時間作為分母。

In [10]:
weather_summary = summarize_category('天候', df['weather_label'])

**天候**｜全樣本 N=22,762；有效 N=22,760；缺值 N=2。百分比分母為 22,760 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,晴,12966,56.9684
1,陰,6440,28.2953
2,雨,3353,14.7320
3,風,1,0.0044


**結果解讀：** 有效紀錄為 22,760 筆、缺值 2 筆。晴天最多（12,966 筆），其次為陰天（6,440 筆）與雨天（3,353 筆），另有 1 筆「風」。這不代表晴天較危險，也可能與晴天較常見或交通量較大有關。

<a id="category-road"></a>

### 2.6 道路型態

此欄區分交岔路、直路、橋樑等型態，數字代碼沒有可直接平均的量尺意義。以下列出全部 14 個已觀察到的類別，讓較少見的型態也能被檢視。

In [11]:
road_summary = summarize_category('道路型態', df['road_form_label'])

**道路型態**｜全樣本 N=22,762；有效 N=22,750；缺值 N=12。百分比分母為 22,750 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,四岔路,8286,36.4220
1,直路,7528,33.0901
2,三岔路,4503,19.7934
3,多岔路,1278,5.6176
4,其他,389,1.7099
5,彎曲路及附近,298,1.3099
6,橋樑,242,1.0637
7,高架道路,68,0.2989
8,坡路,51,0.2242
9,圓環,41,0.1802


**結果解讀：** 有效紀錄為 22,750 筆、缺值 12 筆。四岔路（8,286 筆）與直路（7,528 筆）最多，其次為三岔路（4,503 筆）。件數較少的型態在後續分組比較時，可能只有很少的有效觀測。

<a id="category-solar-night"></a>

### 2.7 日出日落夜間指標

這是主要夜間定義：事故早於當日日出或晚於當日日落為夜間，其餘為非夜間。表格將 0／1 轉為可讀標籤；缺座標的紀錄仍是未知，不歸入非夜間。

In [12]:
solar_night_summary = summarize_category('日出日落夜間指標', df['is_night'].map({0:'非夜間', 1:'夜間'}))

**日出日落夜間指標**｜全樣本 N=22,762；有效 N=22,760；缺值 N=2。百分比分母為 22,760 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,非夜間,15965,70.1450
1,夜間,6795,29.8550


**結果解讀：** 有效紀錄為 22,760 筆，其中夜間 6,795 筆、非夜間 15,965 筆；另有 2 筆無法判定。第4節夜間比較使用這6,795筆，再依照明設備狀態分組。

<a id="category-fixed-night"></a>

### 2.8 固定時段夜間指標

採全年最晚日落後與最早日出前的保守時段：18:49（含）至翌日 05:02（不含）。界線來自資料座標範圍四角全年計算，細節見前述方法定義。套用時只需事故時間，全年使用相同界線；值為0表示不在固定時段內，不等於白天。

In [13]:
fixed_night_summary = summarize_category('固定時段夜間指標', df['night_fixed'].map({0:'固定夜間時段之外', 1:'18:49–05:02'}))

**固定時段夜間指標**｜全樣本 N=22,762；有效 N=22,762；缺值 N=0。百分比分母為 22,762 筆有效紀錄。

,類別,筆數,有效樣本占比（%）
0,固定夜間時段之外,17735,77.9149
1,18:49–05:02,5027,22.0851


**結果解讀：** 固定時段內有 5,027 筆，時段外 17,735 筆，沒有指標缺值。其中 0 筆缺座標而無法逐筆核對；其餘所有入選紀錄都符合當日日出日落夜間。固定時段是較窄的子樣本，不等於完整夜間樣本。

<a id="lighting-night"></a>

## 3. 照明設備狀態與日夜交叉分析

<!-- The actual analysis starts below -->

全年分布說明了設備狀態的構成，接著將三種狀態與日出日落的非夜間／夜間交叉，
才能分辨哪些設備狀態發生於夜間。以下依序呈現3×2件數、各格占完整樣本比例，及目標條件的不同分母比例。

In [14]:
TABLE_DIR = PROJECT_ROOT / 'finding/assets/descriptive_statistics'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
def export_table(name, title, frame):
    show(title, frame)
    frame.to_csv(TABLE_DIR / name, index=False, encoding='utf-8-sig', lineterminator='\n')

<a id="cross-distribution"></a>

### 3.1 交叉件數、占比與樣本分母

這張表把三種照明設備狀態作為列，日出日落指標的「非夜間（0）」與「夜間（1）」作為欄。
本次使用 `is_night`，不使用固定時段指標；非夜間依既定定義包含恰好等於日出或日落的時刻。
先以件數看六個交叉類別，再用有效樣本百分比呈現各格占全部可判定事故的比例。

要計算的目標條件為：
**（有照明未開啟或故障 OR 無照明）AND 夜間**。
OR 只連接兩種照明狀態，夜間條件同時適用兩者；設備未開啟與故障仍無法由原始代碼進一步拆開。

主分析只使用照明狀態與日夜均可判定的紀錄，缺值另列，不視為非夜間或設備已開啟。
為避免「比例」分母含糊，另列三個不同問題：目標事故占所有完整紀錄的比例、
占照明已知的夜間事故比例，以及在上述兩種設備狀態中發生於夜間的比例。
另附已辨識目標紀錄占原始全部紀錄的比例，明示其包含未能分類的樣本，不能當成完整分類下的真實占比。

In [15]:
# 只使用既有的照明與日出日落欄位；不改動資料及夜間定義。
lighting_order = ['有照明且開啟', '有照明未開啟或故障', '無照明']
complete_mask = df['lighting_label'].isin(lighting_order) & df['is_night'].isin([0, 1])
cross_sample = df.loc[complete_mask, ['lighting_label', 'is_night']].copy()
cross_counts = pd.crosstab(cross_sample['lighting_label'], cross_sample['is_night'])
cross_counts = cross_counts.reindex(index=lighting_order, columns=[0, 1], fill_value=0)
cross_counts.columns = ['非夜間（0）', '夜間（1）']
cross_counts.index.name = '照明設備狀態'
complete_n = len(cross_sample)
assert cross_counts.shape == (3, 2)
assert int(cross_counts.to_numpy().sum()) == complete_n

sample_audit = pd.DataFrame([
    {'項目': '原始全部事故紀錄', '筆數': len(df)},
    {'項目': '照明與日夜皆可判定（3×2有效樣本）', '筆數': complete_n},
    {'項目': '至少一欄不可判定（不重複計數）', '筆數': int((~complete_mask).sum())},
    {'項目': '照明缺值', '筆數': int(df['lighting_label'].isna().sum())},
    {'項目': '日夜缺值（與照明缺值可能重疊）', '筆數': int(df['is_night'].isna().sum())},
    {'項目': '兩欄同時缺值', '筆數': int(df[['lighting_label','is_night']].isna().all(axis=1).sum())},
])
show('交叉樣本與缺值核對（缺值項目可能重疊，不能直接加總）', sample_audit)
export_table('04a_lighting_night_counts.csv',
             f'照明設備狀態 × 日出日落夜間：事故件數（有效 N={complete_n:,}）',
             cross_counts.reset_index())

cross_percent = cross_counts / complete_n * 100
assert np.isclose(cross_percent.to_numpy().sum(), 100)
export_table('04b_lighting_night_joint_percent.csv',
             f'六格各占完整樣本比例（%；每格分母均為 {complete_n:,}）',
             cross_percent.reset_index())

target_groups = ['有照明未開啟或故障', '無照明']
target_count = int(cross_counts.loc[target_groups, '夜間（1）'].sum())
night_known_count = int(cross_counts['夜間（1）'].sum())
off_or_absent_count = int(cross_counts.loc[target_groups].to_numpy().sum())
# 用原始代碼獨立核對目標條件，防止只將夜間條件套到其中一組。
assert target_count == int((df['光線'].isin([6, 7]) & df['is_night'].eq(1)).sum())

target_ratios = pd.DataFrame([
    {'比例名稱': '目標事故占所有完整紀錄', '分子（件）': target_count,
     '分母（件）': complete_n, '分母定義': '照明與日夜皆可判定的全部事故',
     '比例（%）': target_count / complete_n * 100},
    {'比例名稱': '夜間事故中上述兩種設備狀態占比', '分子（件）': target_count,
     '分母（件）': night_known_count, '分母定義': '照明可判定的夜間事故',
     '比例（%）': target_count / night_known_count * 100},
    {'比例名稱': '上述兩種設備狀態中夜間事故占比', '分子（件）': target_count,
     '分母（件）': off_or_absent_count, '分母定義': '未開啟或故障、或無照明且日夜可判定的事故',
     '比例（%）': target_count / off_or_absent_count * 100},
    {'比例名稱': '已辨識目標紀錄占原始全部紀錄（含缺值）', '分子（件）': target_count,
     '分母（件）': len(df), '分母定義': '原始全部事故；不可判定者仍含在分母',
     '比例（%）': target_count / len(df) * 100},
])
export_table('04c_target_night_lighting_proportions.csv',
             f'目標條件的比例與分母（目標事故 {target_count:,} 件）', target_ratios)

display(Markdown(
    f'**結果：** 夜間設備未開啟或故障 {int(cross_counts.loc[target_groups[0], "夜間（1）"]):,} 件，'
    f'夜間無照明 {int(cross_counts.loc[target_groups[1], "夜間（1）"]):,} 件，合計 **{target_count:,} 件**。'
    f'占全部完整紀錄 **{target_count:,} / {complete_n:,} = {target_count/complete_n*100:.2f}%**；'
    f'占照明已知的夜間事故 **{target_count:,} / {night_known_count:,} = {target_count/night_known_count*100:.2f}%**。'
    f'在上述兩種設備狀態中，夜間事故占 {target_count:,} / {off_or_absent_count:,} = '
    f'{target_count/off_or_absent_count*100:.2f}%。'
))
verify(data_path, EXPECTED_SHA256)
print('3×2件數加總、百分比分母與原始照明代碼條件核對通過；三份交叉分析CSV已保存。')

### 交叉樣本與缺值核對（缺值項目可能重疊，不能直接加總）

,項目,筆數
0,原始全部事故紀錄,22762
1,照明與日夜皆可判定（3×2有效樣本）,22750
2,至少一欄不可判定（不重複計數）,12
3,照明缺值,12
4,日夜缺值（與照明缺值可能重疊）,2
5,兩欄同時缺值,2


### 照明設備狀態 × 日出日落夜間：事故件數（有效 N=22,750）

,照明設備狀態,非夜間（0）,夜間（1）
0,有照明且開啟,2726,6332
1,有照明未開啟或故障,8933,363
2,無照明,4299,97


### 六格各占完整樣本比例（%；每格分母均為 22,750）

,照明設備狀態,非夜間（0）,夜間（1）
0,有照明且開啟,11.9824,27.8330
1,有照明未開啟或故障,39.2659,1.5956
2,無照明,18.8967,0.4264


### 目標條件的比例與分母（目標事故 460 件）

,比例名稱,分子（件）,分母（件）,分母定義,比例（%）
0,目標事故占所有完整紀錄,460,22750,照明與日夜皆可判定的全部事故,2.0220
1,夜間事故中上述兩種設備狀態占比,460,6792,照明可判定的夜間事故,6.7727
2,上述兩種設備狀態中夜間事故占比,460,13692,未開啟或故障、或無照明且日夜可判定的事故,3.3596
3,已辨識目標紀錄占原始全部紀錄（含缺值）,460,22762,原始全部事故；不可判定者仍含在分母,2.0209


**結果：** 夜間設備未開啟或故障 363 件，夜間無照明 97 件，合計 **460 件**。占全部完整紀錄 **460 / 22,750 = 2.02%**；占照明已知的夜間事故 **460 / 6,792 = 6.77%**。在上述兩種設備狀態中，夜間事故占 460 / 13,692 = 3.36%。

3×2件數加總、百分比分母與原始照明代碼條件核對通過；三份交叉分析CSV已保存。


### 3.2 結果解讀

目標事故是 363 件「有照明未開啟或故障且為夜間」加上 97 件「無照明且為夜間」，
合計 460 件。主要聯合比例為 460 ÷ 22,750 ≈ **2.02%**。
若問題是「夜間事故之中有多少屬於上述兩種設備狀態」，分母改為 6,792，結果為 **6.77%**。
若問題是「上述兩種設備狀態的事故有多少發生於夜間」，分母為 13,692，結果為 **3.36%**。
這三個比例描述不同條件，不能交換使用。

原始資料共 22,762 筆，其中 12 筆照明缺值，這 12 筆又包含 2 筆日夜未知。
因此完整交叉表排除的是 **12 筆，不是 14 筆**。完整夜間紀錄 6,792 筆，
與主要夜間樣本 6,795 筆相差的 3 筆是夜間但照明缺值。
460 ÷ 22,762 的已辨識紀錄占比同樣四捨五入為 2.02%，但分母含未能分類者，
未知紀錄可能還有符合目標條件者，不能把未知條件當成不符合。

本表只描述已發生事故中的設備狀態與日夜分布，沒有交通暴露量，不能用來估計事故發生風險，
也不能把設備未開啟或故障、或無照明解讀為事故原因。此處呈現描述性交叉分布，未進行獨立性檢定。

<a id="night-comparison"></a>

## 4. 夜間不同照明條件比較

交叉分析辨識出夜間設備未開啟或故障363件、無照明97件。
接著限定日出日落夜間6,795筆，比較三個設備組的傷亡與道路環境；其中3筆照明缺值不歸入三組。

### 4.1 傷亡與速限

死亡比例、平均受傷人數與速限各使用該欄有效N。標準差反映事故間差異，不是平均數的估計誤差。
例如無照明組2件死亡事故除以97件，死亡事故比例約2.06%。

In [16]:
# 夜間組別比較：保留設備開啟、未開啟或故障、無設備三組。
night = df.loc[df['is_night'].eq(1)].copy()
GROUPS = ['有照明且開啟','有照明未開啟或故障','無照明']
coverage = night['lighting_label'].fillna('缺值').value_counts()
show(f'夜間樣本照明分類核對（夜間 N={len(night):,}；夜間無法判定 N={df["is_night"].isna().sum()}）',
     coverage.rename_axis('照明條件').reset_index(name='事故件數'))
comparison = []
for g in GROUPS:
    sub = night.loc[night['lighting_label'].eq(g)]
    row = {'照明條件':g,'事故件數':len(sub),'占全部夜間（%）':len(sub)/len(night)*100,
           '死亡有效 N':sub['fatal24'].count(),'死亡缺值 N':sub['fatal24'].isna().sum(),
           '死亡事故件數':sub['fatal24'].sum(min_count=1),'死亡事故比例（%）':sub['fatal24'].mean()*100,
           '死亡指標標準差':sub['fatal24'].std()}
    for col, prefix in [('受傷人數','每件受傷人數'),('速限-速度限制','速限（km/h）')]:
        row.update({prefix+'平均':sub[col].mean(),prefix+'標準差':sub[col].std(),
                    prefix+'有效 N':sub[col].count(),prefix+'缺值 N':sub[col].isna().sum()})
    row['速限為0筆數'] = sub['速限-速度限制'].eq(0).sum()
    comparison.append(row)
comparison = pd.DataFrame(comparison)
# 轉置顯示，使各組及所有分母能在 Notebook 中完整閱讀。
export_table('03a_night_comparison.csv',f'夜間不同照明條件比較（夜間 N={len(night):,}）',
             comparison.set_index('照明條件').T.rename_axis('統計量').reset_index())

### 夜間樣本照明分類核對（夜間 N=6,795；夜間無法判定 N=2）

,照明條件,事故件數
0,有照明且開啟,6332
1,有照明未開啟或故障,363
2,無照明,97
3,缺值,3


### 夜間不同照明條件比較（夜間 N=6,795）

照明條件,統計量,有照明且開啟,有照明未開啟或故障,無照明
0,事故件數,"6,332.0000",363.0000,97.0000
1,占全部夜間（%）,93.1862,5.3422,1.4275
2,死亡有效 N,"6,332.0000",363.0000,97.0000
3,死亡缺值 N,0.0000,0.0000,0.0000
4,死亡事故件數,20.0000,4.0000,2.0000
5,死亡事故比例（%）,0.3159,1.1019,2.0619
6,死亡指標標準差,0.0561,0.1045,0.1428
7,每件受傷人數平均,1.3525,1.3361,1.3093
8,每件受傷人數標準差,0.6400,0.6285,0.6014
9,每件受傷人數有效 N,"6,332.0000",363.0000,97.0000


設備已開啟、未開啟或故障、無設備三組分別為6,332、363、97件，死亡事故比例依序約0.316%、1.102%、2.062%，
每件平均受傷人數約1.352、1.336、1.309人。三組規模差異大，無照明組死亡事件只有2件，
少數事件便會使比例明顯變動；這些是尚未調整環境差異的描述。

### 4.2 速限、天候與道路型態分布

以下各格為「件數（有效組內占比）」，分母為該照明組中該變數非缺值的紀錄。
速限保留原始0值，不視為實際行車速度；各組環境組成的差異可能與傷亡差異同時存在。

In [17]:
# 夜間道路與天候：速限逐值、天候與道路型態；所有觀察到的類別，不只前十類。
distribution = []
for g in GROUPS:
    sub = night.loc[night['lighting_label'].eq(g)]
    for col, label in [('速限-速度限制','速限（km/h；原始值）'),('weather_label','天候'),('road_form_label','道路型態')]:
        valid_n = sub[col].notna().sum()
        counts = sub[col].value_counts(dropna=False).sort_index()
        for category, count in counts.items():
            distribution.append({'照明條件':g,'變數':label,'類別':'缺值' if pd.isna(category) else str(category),
                                 '事故件數':count,'組別 N':len(sub),'有效 N':valid_n,
                                 '缺值 N':sub[col].isna().sum(),
                                 '有效組內占比（%）':count/valid_n*100 if pd.notna(category) and valid_n else np.nan})
distribution = pd.DataFrame(distribution)
for label, sub in distribution.groupby('變數',sort=False):
    # 每組以「件數（有效占比）」並排，另列有效與缺值分母。
    sub = sub.copy()
    sub['件數（有效占比）'] = sub.apply(lambda r: f'{int(r["事故件數"]):,}（{r["有效組內占比（%）"]:.2f}%）'
                                             if pd.notna(r['有效組內占比（%）']) else f'{int(r["事故件數"]):,}（缺值）',axis=1)
    pivot = sub.pivot(index='類別',columns='照明條件',values='件數（有效占比）').reindex(columns=GROUPS).fillna('0（0.00%）')
    show('夜間'+label+'分布（分母見下表；缺值不列入有效百分比）',pivot)
    show(label+'分母',sub[['照明條件','組別 N','有效 N','缺值 N']].drop_duplicates().reset_index(drop=True))
distribution.to_csv(TABLE_DIR/'03b_night_environment_distribution.csv',index=False,encoding='utf-8-sig',lineterminator='\n')
for (g,v), sub in distribution.groupby(['照明條件','變數']):
    assert sub['事故件數'].sum() == len(night.loc[night['lighting_label'].eq(g)])

### 夜間速限（km/h；原始值）分布（分母見下表；缺值不列入有效百分比）

照明條件,有照明且開啟,有照明未開啟或故障,無照明
類別,,,
0.0,86（1.36%）,5（1.38%）,0（0.00%）
20.0,7（0.11%）,0（0.00%）,0（0.00%）
25.0,3（0.05%）,0（0.00%）,0（0.00%）
30.0,"1,157（18.27%）",72（19.83%）,24（24.74%）
40.0,640（10.11%）,37（10.19%）,4（4.12%）
50.0,"4,356（68.79%）",248（68.32%）,68（70.10%）
60.0,52（0.82%）,1（0.28%）,1（1.03%）
70.0,22（0.35%）,0（0.00%）,0（0.00%）
80.0,9（0.14%）,0（0.00%）,0（0.00%）


### 速限（km/h；原始值）分母

,照明條件,組別 N,有效 N,缺值 N
0,有照明且開啟,6332,6332,0
1,有照明未開啟或故障,363,363,0
2,無照明,97,97,0


### 夜間天候分布（分母見下表；缺值不列入有效百分比）

...

,照明條件,組別 N,有效 N,缺值 N
0,有照明且開啟,6332,6332,0
1,有照明未開啟或故障,363,363,0
2,無照明,97,97,0


### 夜間道路型態分布（分母見下表；缺值不列入有效百分比）

照明條件,有照明且開啟,有照明未開啟或故障,無照明
類別,,,
三岔路,"1,191（18.81%）",71（19.56%）,17（17.53%）
其他,109（1.72%）,5（1.38%）,2（2.06%）
四岔路,"2,422（38.25%）",126（34.71%）,36（37.11%）
圓環,13（0.21%）,1（0.28%）,0（0.00%）
地下道,11（0.17%）,1（0.28%）,0（0.00%）
坡路,15（0.24%）,1（0.28%）,1（1.03%）
多岔路,419（6.62%）,22（6.06%）,1（1.03%）
廣場,1（0.02%）,0（0.00%）,0（0.00%）
彎曲路及附近,60（0.95%）,9（2.48%）,4（4.12%）


### 道路型態分母

,照明條件,組別 N,有效 N,缺值 N
0,有照明且開啟,6332,6332,0
1,有照明未開啟或故障,363,363,0
2,無照明,97,97,0


三組速限皆以50 km/h最多，道路型態以四岔路與直路為主。
晴天占比依設備已開啟、未開啟或故障、無設備三組約為44.52%、52.89%、52.58%。
傷亡比較尚未控制這些道路及天候組成，不能直接歸因於照明。

### 4.3 主要兩組的平均差

主要比較「無照明減設備已開啟」，正值表示無照明組平均較高。
死亡指標以百分點呈現，受傷人數與速限保留原始單位。Welch t檢定不假設變異數相同，
但仍依賴紀錄獨立及平均數近似常態；死亡事件稀少，近似推論須審慎。

In [18]:
# 課程要求的兩組平均比較：主要比較無設備 vs 設備已開啟；第三組仍獨立描述。
from scipy.stats import ttest_ind
welch = []
for col,label,scale in [('fatal24','死亡事故比例（百分點）',100),('受傷人數','每件事故受傷人數（人）',1),
                        ('速限-速度限制','速限（km/h；保留0）',1)]:
    a = night.loc[night['lighting_label'].eq('無照明'),col].dropna()*scale
    b = night.loc[night['lighting_label'].eq('有照明且開啟'),col].dropna()*scale
    test = ttest_ind(a,b,equal_var=False) if len(a)>1 and len(b)>1 else None
    welch.append({'指標':label,'無照明有效 N':len(a),'無照明平均':a.mean(),'無照明標準差':a.std(),
                  '開啟有效 N':len(b),'開啟平均':b.mean(),'開啟標準差':b.std(),
                  '平均差（無照明−開啟）':a.mean()-b.mean(),
                  'Welch t 檢定 p 值':float(test.pvalue) if test is not None else np.nan})
export_table('03c_welch_comparison.csv','無照明與已開啟照明的平均差',pd.DataFrame(welch))

### 無照明與已開啟照明的平均差

,指標,無照明有效 N,無照明平均,無照明標準差,開啟有效 N,開啟平均,開啟標準差,平均差（無照明−開啟）,Welch t 檢定 p 值
0,死亡事故比例（百分點）,97,2.0619,14.2842,6332,0.3159,5.6117,1.7460,0.2321
1,每件事故受傷人數（人）,97,1.3093,0.6014,6332,1.3525,0.6400,-0.0432,0.4845
2,速限（km/h；保留0）,97,44.7423,8.7907,6332,44.8050,9.7447,-0.0627,0.9447


死亡比例的平均差約為1.746個百分點，Welch p值約0.2321。
不能把描述上的較高比例當作已確認的統計差異，也不代表已證明沒有差異。
多個指標未做多重比較校正，這裡僅作探索性描述。

<a id="night-sensitivity"></a>

## 5. 夜間定義敏感度

日出日落指標反映日期與位置的變化；固定18:49（含）至翌日05:02（不含）則選取較保守的夜間子樣本。
固定時段5,027筆，其中2筆照明缺值，三個設備組合計5,025筆。
這是重新選取不同事故樣本，不是將同一批紀錄換一個標籤。

In [19]:
sensitivity = []
for definition,mask in [('日出日落',df['is_night'].eq(1)),('固定18:49–05:02',df['night_fixed'].eq(1))]:
    selected = df.loc[mask]
    for g in GROUPS:
        s=selected.loc[selected['lighting_label'].eq(g)]
        sensitivity.append({'夜間定義':definition,'照明條件':g,'事故件數':len(s),
                            '死亡有效 N':s['fatal24'].count(),'死亡事故比例（%）':s['fatal24'].mean()*100,
                            '受傷有效 N':s['受傷人數'].count(),'每件平均受傷人數':s['受傷人數'].mean()})
export_table('03d_night_definition_sensitivity.csv','夜間定義敏感度（逐項有效 N）',pd.DataFrame(sensitivity))

### 夜間定義敏感度（逐項有效 N）

,夜間定義,照明條件,事故件數,死亡有效 N,死亡事故比例（%）,受傷有效 N,每件平均受傷人數
0,日出日落,有照明且開啟,6332,6332,0.3159,6332,1.3525
1,日出日落,有照明未開啟或故障,363,363,1.1019,363,1.3361
2,日出日落,無照明,97,97,2.0619,97,1.3093
3,固定18:49–05:02,有照明且開啟,4746,4746,0.3371,4746,1.3601
4,固定18:49–05:02,有照明未開啟或故障,225,225,1.7778,225,1.3111
5,固定18:49–05:02,無照明,54,54,1.8519,54,1.3333


固定時段下，設備已開啟4,746件、未開啟或故障225件、無設備54件，死亡事故比例約0.337%、1.778%、1.852%。
各組有效分母隨選樣定義改變，引用比例時應同時標明夜間規則與件數。
較窄的固定時段可檢查結果對樣本定義的敏感度，但本身不會消除道路、天候或交通暴露差異。

<a id="presentation-brief"></a>

## 6. 研究重點與簡報引用

全年樣本分布、照明與日夜交叉，以及夜間傷亡比較共同顯示：設備狀態須與事故時段一併閱讀，
且夜間照明組的規模與環境構成不同。交叉分析的目標事故共460件；
2.02%、6.77%與3.36%分別回答聯合比例及兩個不同條件比例，不能互換。

下表由資料直接計算，集中列出件數、人數、百分比及其分母，方便製作簡報時引用。
建議依「資料與定義 → 樣本分布 → 日夜交叉 → 夜間比較 → 敏感度與限制」敘述研究結果。

In [20]:
briefing = []
def brief_count(item, value, unit, sample):
    briefing.append({'指標':item, '數值':value, '單位':unit,
                     '分子':np.nan, '分母':np.nan, '樣本或分母說明':sample})
brief_count('全年事故紀錄',len(df),'件','第一當事人紀錄；暫以每列一事故')
brief_count('24小時內死亡人數（已知值加總）',df['死亡人數'].sum(min_count=1),'人',
            f'有效N={df["死亡人數"].count():,}；缺值N={df["死亡人數"].isna().sum()}')
brief_count('24小時內死亡事故',df['fatal24'].sum(min_count=1),'件',
            f'死亡指標非缺值N={df["fatal24"].count():,}')
brief_count('2–30日死亡人數（已知值加總）',df['2-30日死亡人數'].sum(min_count=1),'人',
            f'有效N={df["2-30日死亡人數"].count():,}；缺值N={df["2-30日死亡人數"].isna().sum()}')
brief_count('受傷人數（已知值加總）',df['受傷人數'].sum(min_count=1),'人',
            f'有效N={df["受傷人數"].count():,}；缺值N={df["受傷人數"].isna().sum()}')
brief_count('日出日落主要夜間',df['is_night'].eq(1).sum(),'件','日夜未知2筆；此列未排除照明缺值')
brief_count('保守固定夜間18:49–05:02',df['night_fixed'].eq(1).sum(),'件','其中2筆照明缺值')
brief_count('照明×日夜完整樣本',complete_n,'件','排除12筆任一分類未知的紀錄，缺值重疊不重複加總')
brief_count('未開啟或故障、或無照明且為夜間',target_count,'件','363件未開啟或故障＋97件無照明')
for _,row in target_ratios.iloc[:3].iterrows():
    briefing.append({'指標':row['比例名稱'],'數值':row['比例（%）'],'單位':'%',
                     '分子':row['分子（件）'],'分母':row['分母（件）'],
                     '樣本或分母說明':row['分母定義']})
briefing = pd.DataFrame(briefing)
export_table('05_presentation_key_numbers.csv','簡報重點數字（依本次實際輸入計算）',briefing)

### 簡報重點數字（依本次實際輸入計算）

,指標,數值,單位,分子,分母,樣本或分母說明
0,全年事故紀錄,"22,762.0000",件,NaN,NaN,第一當事人紀錄；暫以每列一事故
1,24小時內死亡人數（已知值加總）,58.0000,人,NaN,NaN,"有效N=22,755；缺值N=7"
2,24小時內死亡事故,57.0000,件,NaN,NaN,"死亡指標非缺值N=22,755"
3,2–30日死亡人數（已知值加總）,3.0000,人,NaN,NaN,"有效N=22,752；缺值N=10"
4,受傷人數（已知值加總）,"30,201.0000",人,NaN,NaN,"有效N=22,761；缺值N=1"
5,日出日落主要夜間,"6,795.0000",件,NaN,NaN,日夜未知2筆；此列未排除照明缺值
6,保守固定夜間18:49–05:02,"5,027.0000",件,NaN,NaN,其中2筆照明缺值
7,照明×日夜完整樣本,"22,750.0000",件,NaN,NaN,排除12筆任一分類未知的紀錄，缺值重疊不重複加總
8,未開啟或故障、或無照明且為夜間,460.0000,件,NaN,NaN,363件未開啟或故障＋97件無照明
9,目標事故占所有完整紀錄,2.0220,%,460.0000,"22,750.0000",照明與日夜皆可判定的全部事故


<a id="limitations"></a>

## 7. 研究限制與後續工作

- 目前將第一當事人紀錄暫作事故件數，仍須核對官方事故識別碼；候選鍵無重複不等於已獲官方確認。
- 各變數缺值不同，使用各自有效N；未知條件不能直接視為不符合，傷亡總數只是已知值加總。
- 日出日落為Astral天文模型的推算，未納入現場亮度、地形與建築遮蔽。事故時間僅到分鐘，邊界仍有精度限制。
  固定時段適用本年度與資料地理範圍，時段之外不等於白天。
- 設備「未開啟或故障」無法再細分；沒有重傷變數，不能把一般受傷人數稱為重傷人數。
- 速限0與少見值（例如39）保留原值，尚未擅自更正；速限也不是事故當下車速。
- 無照明夜間樣本97件、死亡事件2件，估計容易受少數事件影響。Welch推論僅供探索，未做多重比較校正。
- 缺少交通量與其他暴露資料，只能描述已發生事故中的分布與傷亡比例，不能稱為事故發生率或因果效果。

後續應先核對事件單位、資料品質及重傷定義，再評估是否有足夠樣本控制道路與天候等可觀察特徵，
研究定位仍為關聯分析。

<a id="data-checks"></a>

## 8. 資料驗證與輸出檔案

### 8.1 Assertions

下列檢查確認第一當事人序號、候選鍵、缺值保留規則、月份涵蓋與兩種夜間指標。
全數通過僅代表這些特定檢查通過，不取代官方資料核實；若檢查停止，應先查明原因。

In [21]:
assert df['當事人序號'].eq(1).all()
candidate_key = ['發生年度','發生月','發生日','發生時-Hours','發生分','肇事地點','座標-X','座標-Y']
assert not df.duplicated(candidate_key).any()
assert df['is_night'].isna().equals(df[['座標-X','座標-Y']].isna().any(axis=1))
assert df['fatal24'].isna().equals(df['死亡人數'].isna())
assert df.loc[df['死亡人數'].notna(),'fatal24'].eq(df['死亡人數'].dropna().ge(1)).all()
assert set(dt.dt.month) == set(range(1,13))
valid = df['is_night'].notna()
local_dt = pd.to_datetime(df.loc[valid,'crash_datetime']).dt.tz_localize('Asia/Taipei')
rise = pd.to_datetime(df.loc[valid,'sunrise_local'],utc=True)
setting = pd.to_datetime(df.loc[valid,'sunset_local'],utc=True)
assert df.loc[valid,'is_night'].eq((local_dt < rise) | (local_dt > setting)).all()
clock_minute=dt.dt.hour*60+dt.dt.minute
assert df['night_fixed'].eq(clock_minute.ge(1129)|clock_minute.lt(302)).all()
selected_known=df['night_fixed'].eq(1)&df['is_night'].notna()
assert df.loc[selected_known,'is_night'].eq(1).all()
verify(data_path, EXPECTED_SHA256)
print('Schema、列數、追溯流水號、候選鍵、缺值規則、12個月份與夜間指標檢查通過。')

Schema、列數、追溯流水號、候選鍵、缺值規則、12個月份與夜間指標檢查通過。


In [22]:
assert len(category_summaries) == 7
for name, table in category_summaries.items():
    assert table['類別'].is_unique, f'{name} 有重複類別'
print('七個類別變數均已分表顯示全部類別，件數與百分比分母核對通過。')

七個類別變數均已分表顯示全部類別，件數與百分比分母核對通過。


<a id="csv-index"></a>

### 8.2 表格CSV索引

以下檔案對應本文分析。月份、行政區與光線的全年分布直接參考第2節完整類別摘要。
引用表格時請保留標題、樣本、分母與單位。

| 分析內容 | CSV |
|---|---|
| 3.1 交叉件數 | [照明×日夜件數](assets/descriptive_statistics/04a_lighting_night_counts.csv) |
| 3.1 六格完整樣本占比 | [交叉樣本占比](assets/descriptive_statistics/04b_lighting_night_joint_percent.csv) |
| 3.1 目標條件與分母 | [條件比例](assets/descriptive_statistics/04c_target_night_lighting_proportions.csv) |
| 4.1 夜間傷亡與速限 | [夜間三組比較](assets/descriptive_statistics/03a_night_comparison.csv) |
| 4.2 夜間道路與天候 | [道路環境分布](assets/descriptive_statistics/03b_night_environment_distribution.csv) |
| 4.3 主要兩組平均差 | [平均差比較](assets/descriptive_statistics/03c_welch_comparison.csv) |
| 5 夜間定義敏感度 | [敏感度比較](assets/descriptive_statistics/03d_night_definition_sensitivity.csv) |
| 6 簡報引用數字 | [重點數字](assets/descriptive_statistics/05_presentation_key_numbers.csv) |

本Notebook由乾淨Python kernel依序執行，保留實際輸出。原始與衍生資料由固定路徑與SHA-256綁定，
來源處理程序見第1節；本地成果待學生檢閱後自行stage、commit、push。